# 11 — Algorithms, Complexity, and Caching

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
# Provided for this lesson.
import bisect
from collections.abc import Sequence

from lesson import Exercises, ___, run

check = Exercises('advanced/11', {1: '67ee8d8a2e017389', 2: '3df6fc9d95e9bc08', 3: 'd54e89e1ec4f738a', 4: '696ed16a08f821d1', 5: 'bd993eeafe7b047a', 6: 'e3046cb387e554e0', 7: '7a5560ab0fda8ab8', 8: '80375b4b9708e4b5', 9: 'bec1e0c0ea4c4751'}, practice='advanced_03_security_testing_performance', independent='merge_sorted')

## 1. Binary search with bisect

- `bisect` needs a **sorted** sequence; it finds positions in O(log n).
- `bisect_left(xs, x)` is where `x` would go, before any equal items. Check `i < len(xs) and xs[i] == x` to see if it's there.
- `insort(xs, x)` inserts and keeps the list sorted.

In [ ]:
from bisect import bisect_left, insort
xs = [2, 4, 4, 8]
print(bisect_left(xs, 4), bisect_left(xs, 5))
insort(xs, 5)
print(xs)

**Exercise 1.** Where would `6` go in `[1, 3, 5, 7]`?

In [ ]:
from bisect import bisect_left
position = bisect_left([1, 3, 5, 7], ___)
check(1, position)

**Exercise 2.** Return the index of the target, or -1 if absent.

In [ ]:
from bisect import bisect_left
def find_index(xs, target):
    i = bisect_left(xs, target)
    return i if i < len(xs) and xs[i] == target else ___

check(2, (find_index([1, 3, 5], 5), find_index([1, 3, 5], 4)))

**Exercise 3.** Binary search on an unsorted list gives nonsense. What must be true of the input? Write one word.

In [ ]:
answer = ___
check(3, answer)

## 2. Heaps

- `heapq` keeps a list as a min-heap: `heappush` and `heappop` are O(log n), and the smallest comes out first.
- Push tuples `(priority, counter, item)`: the counter breaks ties in insertion order and avoids comparing items.
- `heapq.nlargest(k, xs)` is O(n log k); `heapq.merge(*sorted_lists)` merges lazily.

In [ ]:
import heapq, itertools
heap, counter = [], itertools.count()
for priority, item in [(2, "b"), (1, "a"), (2, "c")]:
    heapq.heappush(heap, (priority, next(counter), item))
print([heapq.heappop(heap)[2] for _ in range(3)])
print(heapq.nlargest(2, [5, 1, 9, 3]), list(heapq.merge([1, 4], [2, 3])))

**Exercise 4.** Pop from a min-heap built from `[5, 1, 3]`. Which comes out first?

In [ ]:
import heapq
h = [5, 1, 3]
heapq.heapify(h)
predicted = ___   # heapq.heappop(h)
check(4, predicted)

**Exercise 5.** Equal priorities come out in insertion order thanks to the counter. Predict the order.

In [ ]:
import heapq, itertools
heap, counter = [], itertools.count()
for p, item in [(1, "x"), (0, "y"), (1, "z")]:
    heapq.heappush(heap, (p, next(counter), item))
order = [heapq.heappop(heap)[2] for _ in range(3)]
predicted = ___
check(5, predicted)

**Exercise 6.** The 3 largest values, largest first.

In [ ]:
import heapq
top = heapq.nlargest(___, [4, 10, 1, 7, 3])
check(6, top)

**Exercise 7.** Merge two sorted lists into one sorted list.

In [ ]:
import heapq
merged = list(heapq.merge([1, 5, 9], ___))
check(7, merged)

## 3. Caching and dedup

- Cache only pure functions (same input → same output, no side effects).
- `set` membership is O(1): dedupe in order with `seen = set()` and a loop, or `dict.fromkeys`.
- A cache needs a size limit and a way to invalidate stale entries.

In [ ]:
def dedupe(items):
    seen, out = set(), []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

print(dedupe([3, 1, 3, 2, 1]))

**Exercise 8.** Dedupe, keeping the first of each in order.

In [ ]:
items = [3, 1, 3, 2, 1]
unique = list(dict.fromkeys(___))
check(8, unique)

**Exercise 9.** Is a function that reads the current time safe to cache? Write `True` or `False`.

In [ ]:
answer = ___
check(9, answer)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### 1. `StablePriorityQueue`

A min-priority queue that is FIFO for equal priorities.

In [ ]:
class StablePriorityQueue:
    """Min-priority queue; items with equal priority come out in insertion (FIFO) order.

    Hint: push (priority, sequence_number, item) tuples onto a heapq list.
    """

    def __init__(self) -> None:
        self._heap: list = []
        self._counter = ___

    def push(self, priority: int, item) -> None:
        heapq.heappush(___, (priority, next(self._counter), item))

    def pop(self):
        if ___:
            raise IndexError("pop from empty queue")
        return heapq.heappop(self._heap)[2]

    def __len__(self) -> int:
        return ___

In [ ]:
check('StablePriorityQueue')

### 2. `find_index`

Binary search with `bisect`; -1 if absent.

In [ ]:
def find_index(sorted_values: Sequence, target) -> int:
    """Binary search with bisect; return the index of target or -1 if absent."""
    index = bisect.bisect_left(sorted_values, target)
    return ___

In [ ]:
check('find_index')

### Independent build — `merge_sorted`

Return one sorted list containing every item of the already-sorted inputs, including duplicates. Accept zero or more sequences, including empty ones.

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
def merge_sorted(*sequences: Sequence) -> list:
    raise NotImplementedError

In [ ]:
check('merge_sorted')

### 4. `top_k`

The k largest values, largest first.

In [ ]:
def top_k(values: Sequence[float], k: int) -> list:
    """The k largest values, largest first, in O(n log k). Hint: heapq.nlargest."""
    return ___

In [ ]:
check('top_k')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** bisect works on…

   - **a)** Any list
   - **b)** Sets
   - **c)** A sorted list

**2.** heapq.heappop returns…

   - **a)** The largest item
   - **b)** The smallest item
   - **c)** The last pushed item

**3.** The hard part of caching changing data is…

   - **a)** Syntax
   - **b)** Hashing
   - **c)** Invalidation

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/11', {1: '9877578a831b', 2: 'a149e53f7ccc', 3: 'dc9d7af7573c'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [12 — Architecture, Ports and Adapters, Dependency Injection](12_architecture_dependency_injection.ipynb)